# Lateral Double Quantum Dot with Top and Horizontal Barriers

This structure is the upper part of `07_square_quantum_dot_array.ipynb`, retained only through its two horizontal separator barriers. The plungers and the three barriers that separate them all enter from the top. `B_H_L` and `B_H_R` run from the outer edge of each ohmic to the outer end of the corresponding plunger head; nothing from the square array below those bars is included.

The notebook writes a nextnano++ input with **strain, Poisson, quantum, and quantum-Poisson all disabled**. The executed verification is structure-only. Potential and hole-density plane/line-cut cells are prepared for a future solver-enabled run, but are not run by default.

## 0. Setup

In [26]:
from pathlib import Path
import json
import re
import sys

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from phidl import Device

REPO_ROOT = next(
    path for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (path / "src").exists() and (path / "notebooks").exists()
)
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from qd_design import (
    SeparatedSquareDotArrayDevice,
    build_simulation_layout,
    make_reference_sige_ge_process_stack,
    plot_layout_spec_2d,
    plot_simulation_layout_3d,
    write_nextnano_input_from_template,
)
import nextnanopp_tools as nnt

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /Users/robertjovanov/code/qpu-design-automation-toolkit


In [27]:
OUTPUT_DIR = REPO_ROOT / "data" / "gds"
GENERATED_INPUT_DIR = REPO_ROOT / "configs" / "robert_inputs" / "generated"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_INPUT_DIR.mkdir(parents=True, exist_ok=True)

STEM = "lateral_double_dot_top_and_horizontal_barriers"
GDS_PATH = OUTPUT_DIR / f"{STEM}.gds"
SVG_PATH = OUTPUT_DIR / f"{STEM}.svg"
LAYOUT_SPEC_PATH = OUTPUT_DIR / f"{STEM}_layout_spec.json"
SIMULATION_LAYOUT_PATH = OUTPUT_DIR / f"{STEM}_simulation_layout.json"
TEMPLATE_INPUT_PATH = REPO_ROOT / "configs" / "robert_inputs" / "double_qd" / "3d" / "Double_Quantum_Dot_3D.in"
GENERATED_INPUT_PATH = GENERATED_INPUT_DIR / "Lateral_Double_Quantum_Dot_Top_and_Horizontal_Barriers.in"

RUNS_DIR = REPO_ROOT / "runs"
RUN_TAG = "structure_only_top_and_horizontal_barriers"
RUN_STRUCTURE = False  # The structure-only device run has already been verified.
BIAS_INDEX = 0
SOLVER_OVERRIDES = {
    "strain": 0,
    "poisson": 0,
    "quantum": 0,
    "quantum_poisson": 0,
}
print("Template exists:", TEMPLATE_INPUT_PATH.exists())
print("Structure-only run enabled:", RUN_STRUCTURE)
print("Solver switches:", SOLVER_OVERRIDES)

Template exists: True
Structure-only run enabled: False
Solver switches: {'strain': 0, 'poisson': 0, 'quantum': 0, 'quantum_poisson': 0}


## 1. Derive the retained structure from notebook 07

In [28]:
reference_square = SeparatedSquareDotArrayDevice(
    name="notebook_07_reference",
    device_y_size_nm=200.0,
    row_gap_nm=0.0,
    ohmic_width_nm=40.0,
    ohmic_length_nm=None,
    barrier_width_nm=40.0,
    barrier_length_nm=140.0,
    plunger_body_width_nm=40.0,
    plunger_body_length_nm=50.0,
    plunger_head_top_width_nm=60.0,
    plunger_head_max_width_nm=100.0,
    plunger_head_height_nm=100.0,
    plunger_upper_taper_height_nm=25.0,
    plunger_lower_taper_height_nm=25.0,
    ohmic_to_barrier_gap_nm=20.0,
    barrier_to_plunger_gap_nm=20.0,
)
reference_square.ensure_built()
reference_elements = {item["name"]: item for item in reference_square.layout_spec()}

# Keep notebook 07's upper row and both horizontal barriers. Renaming makes
# this a standalone two-dot device; translating puts the bars at y = 0..40 nm.
SOURCE_TO_TARGET = {
    "OC_L_2": "OC_L", "B4": "B1", "P3": "P1",
    "B5": "B2", "P4": "P2", "B6": "B3",
    "OC_R_2": "OC_R", "B_SEP_L": "B_H_L", "B_SEP_R": "B_H_R",
}
VOLTAGE_LABELS = {name: f"V_{name}" for name in SOURCE_TO_TARGET.values()}
Y_OFFSET_NM = -reference_square.separator_y_min_nm

def translated_element(source_name, target_name):
    item = dict(reference_elements[source_name])
    item["name"] = target_name
    item["voltage_label"] = VOLTAGE_LABELS[target_name]
    item["polygon_xy_nm"] = [
        [[float(x), float(y) + Y_OFFSET_NM] for x, y in polygon]
        for polygon in item["polygon_xy_nm"]
    ]
    all_points = np.asarray([point for polygon in item["polygon_xy_nm"] for point in polygon])
    item.update({
        "x_min_nm": float(all_points[:, 0].min()),
        "x_max_nm": float(all_points[:, 0].max()),
        "y_min_nm": float(all_points[:, 1].min()),
        "y_max_nm": float(all_points[:, 1].max()),
        "center_x_nm": float(0.5 * (all_points[:, 0].min() + all_points[:, 0].max())),
        "center_y_nm": float(0.5 * (all_points[:, 1].min() + all_points[:, 1].max())),
    })
    return item

layout_spec = [translated_element(source, target) for source, target in SOURCE_TO_TARGET.items()]
elements = {item["name"]: item for item in layout_spec}
display(pd.DataFrame([{k: item[k] for k in ("name", "gate_type", "x_min_nm", "x_max_nm", "y_min_nm", "y_max_nm")} for item in layout_spec]))

,name,gate_type,x_min_nm,x_max_nm,y_min_nm,y_max_nm
0,OC_L,ohmic,-260.0,-220.0,70.0,220.0
1,B1,barrier,-200.0,-160.0,80.0,220.0
2,P1,plunger,-140.0,-40.0,70.0,220.0
3,B2,barrier,-20.0,20.0,80.0,220.0
4,P2,plunger,40.0,140.0,70.0,220.0
5,B3,barrier,160.0,200.0,80.0,220.0
6,OC_R,ohmic,220.0,260.0,70.0,220.0
7,B_H_L,barrier,-260.0,-40.0,0.0,40.0
8,B_H_R,barrier,40.0,260.0,0.0,40.0


In [29]:
expected_names = {"OC_L", "B1", "P1", "B2", "P2", "B3", "OC_R", "B_H_L", "B_H_R"}
assert set(elements) == expected_names
assert elements["B_H_L"]["y_min_nm"] == elements["B_H_R"]["y_min_nm"] == 0.0
assert elements["B_H_L"]["y_max_nm"] == elements["B_H_R"]["y_max_nm"] == 40.0
assert np.isclose(elements["B_H_L"]["x_min_nm"], elements["OC_L"]["x_min_nm"])
assert np.isclose(elements["B_H_L"]["x_max_nm"], elements["P1"]["x_max_nm"])
assert np.isclose(elements["B_H_R"]["x_min_nm"], elements["P2"]["x_min_nm"])
assert np.isclose(elements["B_H_R"]["x_max_nm"], elements["OC_R"]["x_max_nm"])
assert all(elements[name]["y_max_nm"] == 220.0 for name in ("B1", "B2", "B3"))
assert all(elements[name]["y_max_nm"] == 220.0 for name in ("P1", "P2"))
assert not ({"P3", "P4", "B4", "B5", "B6"} & set(elements))
print("Verified upper-row-only geometry and exact horizontal-barrier spans.")

Verified upper-row-only geometry and exact horizontal-barrier spans.


### 1.1 Exact polygon view and standalone PHIDL device

In [30]:
layout_fig = plot_layout_spec_2d(
    layout_spec,
    title="Lateral double dot: top barriers plus left/right horizontal barriers",
)
layout_fig.show()

device = Device(STEM)
for item in layout_spec:
    layer = (item["gds_layer"], item["gds_datatype"])
    for polygon in item["polygon_xy_nm"]:
        device.add_polygon(polygon, layer=layer)

device.write_gds(str(GDS_PATH))
device.write_svg(str(SVG_PATH))
LAYOUT_SPEC_PATH.write_text(json.dumps(layout_spec, indent=2), encoding="utf-8")
print("Device bbox:", device.bbox)
print("GDS:", GDS_PATH)
print("SVG:", SVG_PATH)

Device bbox: [[-260.    0.]
 [ 260.  220.]]
GDS: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/lateral_double_dot_top_and_horizontal_barriers.gds
SVG: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/lateral_double_dot_top_and_horizontal_barriers.svg


## 2. Build and preview the 3D device

In [31]:
process_stack = make_reference_sige_ge_process_stack()
simulation_layout = build_simulation_layout(
    name=f"{STEM}_3d",
    layout_elements=layout_spec,
    process_stack=process_stack,
    x_margin_nm=0.0,
    y_margin_nm=0.0,
)
simulation_layout.write_json(str(SIMULATION_LAYOUT_PATH))
assert len(simulation_layout.patterned_regions) == len(layout_spec)

structure_preview_fig = plot_simulation_layout_3d(
    simulation_layout,
    z_range_nm=(-120.0, 180.0),
    show_polygon_outlines=True,
)
structure_preview_fig.show()
print("Simulation layout:", SIMULATION_LAYOUT_PATH)

Simulation layout: /Users/robertjovanov/code/qpu-design-automation-toolkit/data/gds/lateral_double_dot_top_and_horizontal_barriers_simulation_layout.json


## 3. Write the input with every solver disabled

In [32]:
if not TEMPLATE_INPUT_PATH.exists():
    raise FileNotFoundError(TEMPLATE_INPUT_PATH)

voltage_overrides = {name: 0.0 for name in VOLTAGE_LABELS.values()}
voltage_overrides.update({"V_P1": -3.0, "V_P2": -3.0})
write_nextnano_input_from_template(
    simulation_layout=simulation_layout,
    template_path=TEMPLATE_INPUT_PATH,
    output_path=GENERATED_INPUT_PATH,
    voltage_overrides=voltage_overrides,
)
generated_input = nnt.load_input_file(GENERATED_INPUT_PATH)
nnt.set_input_variables(generated_input, SOLVER_OVERRIDES)
nnt.save_input_file(generated_input, fullpath=GENERATED_INPUT_PATH, overwrite=True)

generated_text = GENERATED_INPUT_PATH.read_text(encoding="utf-8")
solver_values = {
    name: (match.group(1).strip() if (match := re.search(rf"^\${name}\s*=\s*([^#\n]+)", generated_text, re.MULTILINE)) else None)
    for name in SOLVER_OVERRIDES
}
assert solver_values == {name: "0" for name in SOLVER_OVERRIDES}
assert all(f'name = "{name}"' in generated_text for name in expected_names)
print("Generated input:", GENERATED_INPUT_PATH)
print("Verified disabled solvers:", solver_values)

Generated input: /Users/robertjovanov/code/qpu-design-automation-toolkit/configs/robert_inputs/generated/Lateral_Double_Quantum_Dot_Top_and_Horizontal_Barriers.in
Verified disabled solvers: {'strain': '0', 'poisson': '0', 'quantum': '0', 'quantum_poisson': '0'}


## 4. Run structure only

Set `RUN_STRUCTURE = True` only when a fresh structure check is needed. The four solver variables remain fixed at zero.

In [33]:
if RUN_STRUCTURE:
    RUNS_DIR.mkdir(parents=True, exist_ok=True)
    input_file = nnt.run_input_file(
        GENERATED_INPUT_PATH,
        output_root=RUNS_DIR,
        tag=RUN_TAG,
        add_timestamp=True,
        variables=SOLVER_OVERRIDES,
        show_log=True,
        convergenceCheck=False,
        staging_root=GENERATED_INPUT_DIR / "staged_runs",
        keep_staged_input=True,
    )
    OUTPUT_RUN_DIR = nnt.resolve_run_root(nnt.get_output_directory(input_file))
    STRUCTURE_DIR = OUTPUT_RUN_DIR / "Structure"
    assert STRUCTURE_DIR.exists()
    assert (STRUCTURE_DIR / "contacts.vtr").exists()
    assert (STRUCTURE_DIR / "materials.vtr").exists()
    print("Verified structure output:", STRUCTURE_DIR)
else:
    OUTPUT_RUN_DIR = None
    STRUCTURE_DIR = None
    print("Structure run skipped.")

Structure run skipped.


## 5. Analyze the completed Poisson + quantum-Poisson run

The cells below mirror notebook 09's analysis and use the completed horizontal-barrier run directly, independently of the optional structure-only run in section 4:

`runs/Lateral_Double_Quantum_Dot_Top_and_Horizontal_Barriers`

In [34]:
ANALYSIS_RUN_DIR = REPO_ROOT / "runs" / "Lateral_Double_Quantum_Dot_Top_and_Horizontal_Barriers"
ANALYSIS_BIAS_DIR = ANALYSIS_RUN_DIR / "bias_00000"
ANALYSIS_QUANTUM_DIR = ANALYSIS_BIAS_DIR / "Quantum"

if not ANALYSIS_RUN_DIR.exists():
    raise FileNotFoundError(ANALYSIS_RUN_DIR)

JOB_DONE_PATH = ANALYSIS_RUN_DIR / "job_done.txt"
job_done_text = JOB_DONE_PATH.read_text(encoding="utf-8").strip() if JOB_DONE_PATH.exists() else ""
JOB_FINISHED_SUCCESSFULLY = (
    JOB_DONE_PATH.exists()
    and "successfully completed" in job_done_text.lower()
)

job_status = pd.Series(
    {
        "run_directory": str(ANALYSIS_RUN_DIR),
        "job_done_exists": JOB_DONE_PATH.exists(),
        "job_done_message": job_done_text,
        "finished_successfully": JOB_FINISHED_SUCCESSFULLY,
        "bias_directory_exists": ANALYSIS_BIAS_DIR.exists(),
        "quantum_directory_exists": ANALYSIS_QUANTUM_DIR.exists(),
    },
    name="value",
)
display(job_status.to_frame())

if not JOB_FINISHED_SUCCESSFULLY:
    raise RuntimeError(
        f"Run did not report successful completion in {JOB_DONE_PATH}: {job_done_text!r}"
    )

,value
run_directory,/Users/robertjovanov/code/qpu-design-automatio...
job_done_exists,True
job_done_message,Calculation successfully completed.
finished_successfully,True
bias_directory_exists,True
quantum_directory_exists,True


### 5.1 Integrated hole density

In [35]:
INTEGRATED_DENSITY_PATH = ANALYSIS_RUN_DIR / "integrated_density_hole.dat"
integrated_density_hole = nnt.read_integrated_density_hole(INTEGRATED_DENSITY_PATH)
integrated_region_columns = nnt.integrated_density_region_columns(integrated_density_hole)

display(integrated_density_hole)

integrated_density_summary = pd.DataFrame(
    {
        "region": integrated_region_columns,
        "integrated_holes": [
            float(integrated_density_hole[column].iloc[-1])
            for column in integrated_region_columns
        ],
    }
)
integrated_density_summary.loc[len(integrated_density_summary)] = {
    "region": "all reported regions",
    "integrated_holes": integrated_density_summary["integrated_holes"].sum(),
}
display(integrated_density_summary)

,OC_L_bias[V],B1_bias[V],P1_bias[V],B2_bias[V],P2_bias[V],B3_bias[V],OC_R_bias[V],B_H_L_bias[V],B_H_R_bias[V],Body_bias[V],remove_surface_charge_bias[V],zero_fermi_QW_bias[V],region_2[carriers]
0,0,0,0,0,0,0,0,0,0,0,-1.5,0,1.740797e-302


,region,integrated_holes
0,region_2[carriers],1.740797e-302
1,all reported regions,1.740797e-302


### 5.2 Total-charge summary

In [36]:
TOTAL_CHARGES_PATH = ANALYSIS_BIAS_DIR / "total_charges.txt"
total_charges = nnt.read_total_charges(TOTAL_CHARGES_PATH)
display(total_charges)

charge_values = total_charges.set_index("quantity")["value"]
total_charge_summary = pd.Series(
    {
        "electrons_e": float(charge_values.get("electrons", np.nan)),
        "holes_e": float(charge_values.get("holes", np.nan)),
        "ionized_donors_e": float(charge_values.get("ionized donors", np.nan)),
        "ionized_acceptors_e": float(charge_values.get("ionized acceptors", np.nan)),
        "fixed_charges_e": float(charge_values.get("fixed charges", np.nan)),
        "polarization_total_e": float(charge_values.get("polarization_total", np.nan)),
        "reported_sum_e": float(charge_values.get("sum", np.nan)),
    },
    name="value",
)
display(total_charge_summary.to_frame())

,quantity,value,unit
0,electrons,-0.000000e+00,e
1,holes,1.959250e-302,e
2,ionized donors,0.000000e+00,e
3,ionized acceptors,-0.000000e+00,e
4,fixed charges,0.000000e+00,e
5,polarization_total,0.000000e+00,e
6,polarization_piezo,0.000000e+00,e
7,sum,1.959250e-302,e


,value
electrons_e,-0.000000e+00
holes_e,1.959250e-302
ionized_donors_e,0.000000e+00
ionized_acceptors_e,-0.000000e+00
fixed_charges_e,0.000000e+00
polarization_total_e,0.000000e+00
reported_sum_e,1.959250e-302


## 6. Reusable VTR plane- and line-cut extraction

`extract_vtr_cuts` reads a chosen plane and line directly from an ASCII VTR file. Each result is plotted in a separate figure using raw linear values. For the default `z` plane and `x` line, the readers use memory-mapped windows and avoid loading the complete 3D volume.

In [37]:
def extract_vtr_cuts(
    path,
    *,
    variable,
    plane_axis="z",
    plane_value_nm=-3.75,
    line_axis="x",
    line_fixed_coords_nm=None,
):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    available_variables = nnt.list_variables(path)
    if variable not in available_variables:
        raise KeyError(
            f"{variable!r} is not present in {path.name}; "
            f"available variables: {available_variables}"
        )

    if line_fixed_coords_nm is None:
        line_fixed_coords_nm = {"y": 120.0, "z": plane_value_nm}

    plane = nnt.load_vtr_plane(
        path,
        variable=variable,
        slice_axis=plane_axis,
        slice_value=plane_value_nm,
    )
    line = nnt.load_vtr_linecut(
        path,
        variable=variable,
        axis=line_axis,
        fixed_coords=line_fixed_coords_nm,
    )
    return {
        "path": path,
        "variable": variable,
        "available_variables": available_variables,
        "plane": plane,
        "line": line,
    }


def plane_values_yx(plane):
    values = np.asarray(plane["values"], dtype=float)
    if values.shape == (len(plane["x"]), len(plane["y"])):
        return values.T
    if values.shape == (len(plane["y"]), len(plane["x"])):
        return values
    raise ValueError(
        f"Cannot orient plane shape {values.shape} against "
        f"x={len(plane['x'])}, y={len(plane['y'])}."
    )


def plot_vtr_plane(cuts, *, title=None, colorscale="Turbo", color_range=None):
    plane = cuts["plane"]
    values_yx = plane_values_yx(plane)
    variable_label = plane["variable"].label or plane["variable"].name
    zmin, zmax = (None, None) if color_range is None else color_range

    fig = go.Figure(
        go.Heatmap(
            x=plane["x"],
            y=plane["y"],
            z=values_yx,
            colorscale=colorscale,
            zmin=zmin,
            zmax=zmax,
            zsmooth=False,
            connectgaps=False,
            hoverongaps=False,
            colorbar={"title": variable_label},
        )
    )
    fig.update_layout(
        title=title or f"{cuts['path'].name}: {cuts['variable']}",
        xaxis_title=plane["x_label"],
        yaxis_title=plane["y_label"],
        template="plotly_white",
        height=560,
        margin={"l": 60, "r": 50, "t": 75, "b": 55},
        meta={
            "requested_plane_nm": plane.get("requested_slice_coordinate"),
            "actual_plane_nm": plane["slice_coordinate"],
            "linear_scale": True,
        },
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    return fig


def plot_vtr_line(cuts, *, title=None):
    line = cuts["line"]
    variable_label = line["variable"].label or line["variable"].name
    fixed_text = ", ".join(
        f"{name}={value:.3f} nm"
        for name, value in line["chosen_coords"].items()
    )
    fig = go.Figure(
        go.Scatter(
            x=line["axis"],
            y=np.asarray(line["values"], dtype=float),
            mode="lines",
            name=cuts["variable"],
        )
    )
    fig.update_layout(
        title=title or f"{cuts['path'].name}: {cuts['variable']} ({fixed_text})",
        xaxis_title=line["axis_label"],
        yaxis_title=variable_label,
        template="plotly_white",
        height=480,
        showlegend=False,
        margin={"l": 70, "r": 35, "t": 75, "b": 55},
        meta={"chosen_fixed_coordinates_nm": line["chosen_coords"], "linear_scale": True},
    )
    return fig

### 6.1 Requested VTR outputs and exact QW cut coordinates

All planes below are `xy` cuts at exactly `z = -3.750 nm`. The line crosses the translated plunger heads at `y = 120 nm`, the geometry-relative counterpart of notebook 09's `y = 100 nm` cut. All values are shown on linear scales without smoothing or interpolation.

In [38]:
CUT_PLANE_AXIS = "z"
CUT_PLANE_VALUE_NM = -3.75
CUT_LINE_AXIS = "x"
CUT_LINE_FIXED_COORDS_NM = {"y": 120.0, "z": -3.75}

VTR_OUTPUTS = [
    {
        "label": "Hole density (density_hole.vtr)",
        "path": ANALYSIS_BIAS_DIR / "density_hole.vtr",
        "variable": "Hole_density",
        "shared_scale_group": "density",
    },
    {
        "label": "Electrostatic potential (potential.vtr)",
        "path": ANALYSIS_BIAS_DIR / "potential.vtr",
        "variable": "Potential",
        "shared_scale_group": None,
    },
    {
        "label": "Quantum HH density (density_c-Ge_QW_HH.vtr)",
        "path": ANALYSIS_QUANTUM_DIR / "density_c-Ge_QW_HH.vtr",
        "variable": "Density",
        "shared_scale_group": "density",
    },
    *[
        {
            "label": f"Shifted HH probability, state {state}",
            "path": ANALYSIS_QUANTUM_DIR / f"probability_shift_c-Ge_QW_HH_{state:04d}.vtr",
            "variable": f"Psi^2_{state}",
            "shared_scale_group": None,
        }
        for state in (1, 2, 3)
    ],
]

vtr_manifest = pd.DataFrame(
    {
        "label": item["label"],
        "path": str(item["path"]),
        "exists": item["path"].exists(),
        "variable": item["variable"],
        "available_variables": (
            nnt.list_variables(item["path"]) if item["path"].exists() else []
        ),
    }
    for item in VTR_OUTPUTS
)
display(vtr_manifest)
assert vtr_manifest["exists"].all()

,label,path,exists,variable,available_variables
0,Hole density (density_hole.vtr),/Users/robertjovanov/code/qpu-design-automatio...,True,Hole_density,[Hole_density]
1,Electrostatic potential (potential.vtr),/Users/robertjovanov/code/qpu-design-automatio...,True,Potential,[Potential]
2,Quantum HH density (density_c-Ge_QW_HH.vtr),/Users/robertjovanov/code/qpu-design-automatio...,True,Density,[Density]
3,"Shifted HH probability, state 1",/Users/robertjovanov/code/qpu-design-automatio...,True,Psi^2_1,"[E_1, Psi^2_1]"
4,"Shifted HH probability, state 2",/Users/robertjovanov/code/qpu-design-automatio...,True,Psi^2_2,"[E_2, Psi^2_2]"
5,"Shifted HH probability, state 3",/Users/robertjovanov/code/qpu-design-automatio...,True,Psi^2_3,"[E_3, Psi^2_3]"


### 6.2 Separate raw plane and line plots

The classical and quantum density planes share one linear color range so their spatial patterns can be compared directly. The shifted probability fields include their state-energy offsets and are plotted exactly as written by nextnano.

In [39]:
vtr_cut_results = {
    output["label"]: extract_vtr_cuts(
        output["path"],
        variable=output["variable"],
        plane_axis=CUT_PLANE_AXIS,
        plane_value_nm=CUT_PLANE_VALUE_NM,
        line_axis=CUT_LINE_AXIS,
        line_fixed_coords_nm=CUT_LINE_FIXED_COORDS_NM,
    )
    for output in VTR_OUTPUTS
}

density_outputs = [
    output for output in VTR_OUTPUTS
    if output["shared_scale_group"] == "density"
]
SHARED_DENSITY_COLOR_RANGE = (
    0.0,
    max(
        float(np.nanmax(plane_values_yx(vtr_cut_results[output["label"]]["plane"])))
        for output in density_outputs
    ),
)
print("Shared raw density color range:", SHARED_DENSITY_COLOR_RANGE)

cut_consistency_rows = []
for output in VTR_OUTPUTS:
    cuts = vtr_cut_results[output["label"]]
    plane = cuts["plane"]
    line = cuts["line"]
    plane_yx = plane_values_yx(plane)
    line_y_coordinate = line["chosen_coords"].get("y")
    plane_y_index = int(np.argmin(np.abs(np.asarray(plane["y"]) - line_y_coordinate)))
    line_matches_plane = np.array_equal(
        plane_yx[plane_y_index, :],
        np.asarray(line["values"]),
    )
    cut_consistency_rows.append(
        {
            "output": output["label"],
            "actual_plane_z_nm": plane["slice_coordinate"],
            "line_y_nm": line_y_coordinate,
            "line_z_nm": line["chosen_coords"].get("z"),
            "line_matches_plane_row_exactly": line_matches_plane,
            "finite_plane_values": bool(np.isfinite(plane_yx).all()),
        }
    )

    color_range = (
        SHARED_DENSITY_COLOR_RANGE
        if output["shared_scale_group"] == "density"
        else None
    )
    display(
        plot_vtr_plane(
            cuts,
            title=f"{output['label']}: xy at z = -3.750 nm",
            color_range=color_range,
        )
    )
    display(
        plot_vtr_line(
            cuts,
            title=f"{output['label']}: x line at y = 120 nm, z = -3.750 nm",
        )
    )

cut_consistency = pd.DataFrame(cut_consistency_rows)
display(cut_consistency)
assert cut_consistency["line_matches_plane_row_exactly"].all()
assert cut_consistency["finite_plane_values"].all()

Shared raw density color range: (0.0, 9.24752184035e-286)


,output,actual_plane_z_nm,line_y_nm,line_z_nm,line_matches_plane_row_exactly,finite_plane_values
0,Hole density (density_hole.vtr),-3.75,120.0,-3.75,True,True
1,Electrostatic potential (potential.vtr),-3.75,120.0,-3.75,True,True
2,Quantum HH density (density_c-Ge_QW_HH.vtr),-3.75,120.0,-3.75,True,True
3,"Shifted HH probability, state 1",-3.75,120.0,-3.75,True,True
4,"Shifted HH probability, state 2",-3.75,120.0,-3.75,True,True
5,"Shifted HH probability, state 3",-3.75,120.0,-3.75,True,True


### 6.3 Classical/quantum density consistency diagnostic

This compares raw planes rather than rendered colors. The two files represent different physical quantities and need not have identical amplitudes, but they must use the same grid and should have consistent spatial structure.

In [40]:
classical_label = "Hole density (density_hole.vtr)"
quantum_label = "Quantum HH density (density_c-Ge_QW_HH.vtr)"
classical_plane = vtr_cut_results[classical_label]["plane"]
quantum_plane = vtr_cut_results[quantum_label]["plane"]
classical_values = plane_values_yx(classical_plane)
quantum_values = plane_values_yx(quantum_plane)

assert np.array_equal(classical_plane["x"], quantum_plane["x"])
assert np.array_equal(classical_plane["y"], quantum_plane["y"])
assert classical_values.shape == quantum_values.shape

significant = (classical_values > 1e12) & (quantum_values > 1e12)
spatial_correlation = float(
    np.corrcoef(classical_values[significant], quantum_values[significant])[0, 1]
)

classical_max_yx = np.unravel_index(np.argmax(classical_values), classical_values.shape)
quantum_max_yx = np.unravel_index(np.argmax(quantum_values), quantum_values.shape)

def xy_at_yx_index(plane, index_yx):
    iy, ix = index_yx
    return float(plane["x"][ix]), float(plane["y"][iy])


density_consistency_summary = pd.Series(
    {
        "same_x_grid": np.array_equal(classical_plane["x"], quantum_plane["x"]),
        "same_y_grid": np.array_equal(classical_plane["y"], quantum_plane["y"]),
        "same_plane_shape": classical_values.shape == quantum_values.shape,
        "all_classical_pixels_finite": bool(np.isfinite(classical_values).all()),
        "all_quantum_pixels_finite": bool(np.isfinite(quantum_values).all()),
        "classical_max_cm^-3": float(np.max(classical_values)),
        "quantum_HH_max_cm^-3": float(np.max(quantum_values)),
        "classical_max_xy_nm": xy_at_yx_index(classical_plane, classical_max_yx),
        "quantum_max_xy_nm": xy_at_yx_index(quantum_plane, quantum_max_yx),
        "significant_pixel_spatial_correlation": spatial_correlation,
    },
    name="value",
)
display(density_consistency_summary.to_frame())

/Users/robertjovanov/miniconda3/envs/rj_thesis_project/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning:

Mean of empty slice.

/Users/robertjovanov/miniconda3/envs/rj_thesis_project/lib/python3.11/site-packages/numpy/_core/_methods.py:130: RuntimeWarning:

invalid value encountered in divide

/Users/robertjovanov/miniconda3/envs/rj_thesis_project/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning:

Degrees of freedom <= 0 for slice

/Users/robertjovanov/miniconda3/envs/rj_thesis_project/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning:

divide by zero encountered in divide

/Users/robertjovanov/miniconda3/envs/rj_thesis_project/lib/python3.11/site-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning:

invalid value encountered in multiply



,value
same_x_grid,True
same_y_grid,True
same_plane_shape,True
all_classical_pixels_finite,True
all_quantum_pixels_finite,True
classical_max_cm^-3,0.0
quantum_HH_max_cm^-3,0.0
classical_max_xy_nm,"(25.0, 220.0)"
quantum_max_xy_nm,"(-30.0, 220.0)"
significant_pixel_spatial_correlation,NaN


## 7. HH occupation and energy spectrum

In [41]:
QUANTUM_REGION = "c-Ge_QW"
QUANTUM_BAND = "HH"
QUANTUM_KPOINT = "k00000"

occupation = nnt.read_quantum_occupation(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    bias=BIAS_INDEX,
).rename(columns=lambda column: "state" if column == "no." else column)

energy_spectrum = nnt.read_quantum_energy_spectrum(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    kpoint=QUANTUM_KPOINT,
    bias=BIAS_INDEX,
).rename(columns=lambda column: "state" if column == "no." else column)

occupation_column = next(column for column in occupation if column != "state")
energy_column = next(column for column in energy_spectrum if column != "state")
quantum_state_table = energy_spectrum.merge(occupation, on="state", how="outer")

display(quantum_state_table)

occupied_mask = occupation[occupation_column] > 1e-6
quantum_summary = pd.Series(
    {
        "number_of_reported_states": len(quantum_state_table),
        "total_HH_occupation_holes": float(occupation[occupation_column].sum()),
        "states_with_occupation_above_1e-6": int(occupied_mask.sum()),
        "highest_state_with_occupation_above_1e-6": int(
            occupation.loc[occupied_mask, "state"].max()
        ),
        "energy_min_eV": float(energy_spectrum[energy_column].min()),
        "energy_max_eV": float(energy_spectrum[energy_column].max()),
        "states_with_positive_energy": int((energy_spectrum[energy_column] > 0).sum()),
    },
    name="value",
)
display(quantum_summary.to_frame())

,state,Energy[eV],Occupation[holes]
0,1,-0.024707,1.883336e-304
1,2,-0.024938,1.881264e-304
2,3,-0.025516,1.874447e-304
3,4,-0.026143,1.868407e-304
4,5,-0.026735,1.858354e-304
...,...,...,...
95,96,-0.039488,1.780331e-304
96,97,-0.039673,1.769136e-304
97,98,-0.039767,1.732971e-304
98,99,-0.039799,1.706149e-304


ValueError: cannot convert float NaN to integer

In [42]:
occupation_figure = nnt.plot_quantum_occupation(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    bias=BIAS_INDEX,
    interactive=True,
)

energy_spectrum_figure = nnt.plot_quantum_energy_spectrum(
    ANALYSIS_RUN_DIR,
    region=QUANTUM_REGION,
    band=QUANTUM_BAND,
    kpoint=QUANTUM_KPOINT,
    bias=BIAS_INDEX,
    interactive=True,
)